# Random-Forest to Predict Sleep Efficiency

**Goal** Aggregate multimodal epoch-level features **per subject / night**,
compute **Sleep Efficiency (SE%)** from the hypnogram, and train a
Random-Forest regressor to predict SE from the aggregated features.

*Definitions*  
Sleep Efficiency = *(Total Sleep Time / Time in Bed) × 100*  
where **Total Sleep Time (TST)** is any epoch **not** scored Wake (W/0).

*Input CSVs* (created by previous notebooks)
* `sleep_edf_features.csv` EEG band-power (δ θ α σ β) + `stage` + `subject`
* `hrv_features.csv`       SDNN RMSSD LF/HF TINN + `subject`
* `eda_features.csv`       peak_rate … phasic_AUC + `subject`

## 1 – Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt
import seaborn as sns

## 2 – Load EEG / HRV / EDA epoch tables and merge

All three tables must contain two key columns:  
* `subject` – identifier of the night / individual  
* `stage`   – sleep stage per 30-s epoch (0 = Wake, 1-4 = N1…REM)

In [ ]:
EEG_PATH = Path("./outputs/sleep_edf_features.csv")
HRV_PATH = Path("./outputs/hrv_features.csv")
EDA_PATH = Path("./outputs/eda_features.csv")

eeg_df = pd.read_csv(EEG_PATH)
hrv_df = pd.read_csv(HRV_PATH)
eda_df = pd.read_csv(EDA_PATH)

# Quick sanity-check for required columns
for name, df in [("EEG", eeg_df), ("HRV", hrv_df), ("EDA", eda_df)]:
    if not {"subject", "stage"}.issubset(df.columns):
        raise ValueError(f"{name} table missing 'subject' or 'stage' column")

merged = (
    eeg_df
    .merge(hrv_df, on=["epoch", "subject", "stage"])
    .merge(eda_df, on=["epoch", "subject", "stage"])
)
print("Merged epochs:", merged.shape)

## 3 – Compute Sleep Efficiency (label y)

* Time in Bed (TIB)  = number_of_epochs × 30 s  
* Total Sleep Time (TST) = epochs whose stage ≠ Wake (0/‘W’)  
* SE % = (TST / TIB) × 100

In [ ]:
EPOCH_SEC = 30

def se_from_group(df_g):
    tib = len(df_g) * EPOCH_SEC
    tst = (df_g["stage"] != 0).sum() * EPOCH_SEC
    return 100 * tst / tib if tib else np.nan

se_series = merged.groupby("subject").apply(se_from_group).rename("SE")
se_series.head()

## 4 – Aggregate features (mean) per subject

In [ ]:
# Drop non-feature columns
numeric_cols = merged.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in numeric_cols if c not in ("stage",)]

agg_df = (
    merged.groupby("subject")[feature_cols]
    .mean()
    .merge(se_series, left_index=True, right_index=True)
)

print("Subject-level table:", agg_df.shape)
display(agg_df.head())

## 5 – Train / test split

In [ ]:
X = agg_df.drop(columns="SE").values
y = agg_df["SE"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print("Train subjects:", X_train.shape[0], "  Test subjects:", X_test.shape[0])

## 6 – Fit Random-Forest & evaluate

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f"Test MAE  : {mean_absolute_error(y_test, y_pred):.2f} %")
print(f"Test R²   : {r2_score(y_test, y_pred):.2f}")

# Optional: cross-validated R²
cv_r2 = cross_val_score(rf, X, y, cv=5, scoring="r2").mean()
print(f"5-fold CV R²: {cv_r2:.2f}")


## 7 – Permutation importance

Which aggregated features contribute most to SE prediction ?

In [ ]:
result = permutation_importance(
    rf, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1
)
imp_df = (
    pd.DataFrame({
        "feature": agg_df.drop(columns="SE").columns,
        "importance": result.importances_mean,
    })
    .sort_values("importance", ascending=False)
)

plt.figure(figsize=(7, 4))
sns.barplot(data=imp_df.head(10), x="importance", y="feature", color="steelblue")
plt.xlabel("Δ MAE when feature shuffled")
plt.title("Top-10 permutation importances")
plt.tight_layout()
plt.show()

## 8 – Wrap-up

* Subject-level Random-Forest reached **R² ≈ 0.6** (example),
  predicting Sleep Efficiency within ±5–10 %.  
* Key predictors often include **sigma-band power** and **HRV SDNN**,
  hinting at consolidated N2/N3 sleep and autonomic stability.  
* Permutation importance is preferable to built-in `feature_importances_`
  for non-uniform feature scales.  
* Extending to wearable data requires careful handling of artefacts
  and less granular staging (e.g., actigraphy-based sleep/wake).

In [ ]:
agg_df.to_csv("./outputs/subject_level_features.csv", index=True)
print("Saved subject-level table → ./outputs/subject_level_features.csv")